# rank0-only-side-effects — faded example 3: Per-rank computation must live outside the rank-0 guard

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`. The last cell reports your progress on the `Distributed: rank-0-only side effects` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A subtle bug is accidentally placing per-rank computation (like gradient accumulation or local metric recording) inside the `if rank == 0:` block. This makes those operations run only on rank 0, silently dropping contributions from all other ranks. Per-rank computation belongs outside any guard; only shared-resource side effects belong inside `if rank == 0:`.

## Faded exercise 3

### Exercise — Per-rank recording outside the rank-0 guard

Complete `step(rank, world_size, loss, per_rank_fn, log_fn)`. `per_rank_fn` must be called on EVERY rank (outside any guard). `log_fn` must be called only on rank 0.

Fill in the call to per_rank_fn that happens unconditionally.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def step(rank, world_size, loss, per_rank_fn, log_fn):
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    if rank == 0:
        log_fn(f'loss={loss:.4f}')

# Test
recorded = []
logged = []
for r in range(5):
    step(r, 5, 0.1 * r, lambda rk, l: recorded.append(rk), logged.append)
print(recorded, len(logged))


def _test():
    recorded = []
    logged = []
    for r in range(5):
        step(r, 5, 0.1 * r,
             lambda rk, l: recorded.append(rk),
             logged.append)
    assert len(recorded) == 5, f'per_rank_fn must run on all 5 ranks, got {len(recorded)}'
    assert set(recorded) == {0, 1, 2, 3, 4}, 'every rank must record'
    assert len(logged) == 1, f'log_fn must run only once (rank 0), got {len(logged)}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def step(rank, world_size, loss, per_rank_fn, log_fn):
    per_rank_fn(rank, loss)
    if rank == 0:
        log_fn(f'loss={loss:.4f}')

# Test
recorded = []
logged = []
for r in range(5):
    step(r, 5, 0.1 * r, lambda rk, l: recorded.append(rk), logged.append)
print(recorded, len(logged))
```
</details>